In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
from selenium import webdriver
import time
from selenium.webdriver.common.by import By
from selenium.common.exceptions import NoSuchElementException, ElementClickInterceptedException
import os
from datetime import datetime, timedelta
import re
from deep_translator import GoogleTranslator

In [ ]:
# parse the dates

def parse_relative_date(text):
    text = text.lower()
    now = datetime.now()

    if "posted today" in text:
        return now.strftime("%d-%m-%Y")
    elif "posted yesterday" in text:
        return (now - timedelta(days=1)).strftime("%d-%m-%Y")
    
    match_day = re.search(r"posted (\d+) day", text)
    if match_day:
        return (now - timedelta(days=int(match_day.group(1)))).strftime("%d-%m-%Y")

    match_week = re.search(r"posted (\d+) week", text)
    if match_week:
        return (now - timedelta(weeks=int(match_week.group(1)))).strftime("%d-%m-%Y")

    match_month = re.search(r"posted (\d+) month", text)
    if match_month:
        return (now - timedelta(days=30 * int(match_month.group(1)))).strftime("%d-%m-%Y")

    return "Date inconnue"

In [ ]:
# translation

def translate_text(text, source='en', target='fr'):
    try:
        return GoogleTranslator(source=source, target=target).translate(text)
    except:
        return text

In [ ]:
driver = webdriver.Chrome()
driver.get("LINK") #add link
time.sleep(3)

In [ ]:
# list of tender links

while True:
    try:
        load_more = driver.find_element(By.CSS_SELECTOR, "a.load_more_jobs")
        load_more.click()
        time.sleep(2)
    except NoSuchElementException:
        break
    except ElementClickInterceptedException:
        driver.execute_script("window.scrollBy(0, 100);")
        time.sleep(1)
    except Exception as e:
        break

link_list= []

soup = BeautifulSoup(driver.page_source, "html.parser")
tenders = soup.find("ul", class_="job_listings").find_all("li")
for tender in tenders:
    a_tag = tender.find("a")
    if a_tag is None:
        continue
    link= a_tag["href"]
    link_list.append(link)

In [ ]:
# scraping

tenders = []

for l in link_list:
    driver.get(l)
    time.sleep(1)  

    try:
        publish_date = driver.find_element(By.CSS_SELECTOR, "li.date-posted").text
        publish_date = parse_relative_date(publish_date)
    except NoSuchElementException:
        print("tender expired")
        continue

    try:
        div = driver.find_element(By.CLASS_NAME, "job_description")
        strong_tags = div.find_elements(By.TAG_NAME, "strong")
        
        if len(strong_tags) >= 2:
            title = max([tag.text.strip() for tag in strong_tags], key=len)
        else:
            title = strong_tags[0].text.strip() if strong_tags else "No title found"
        translated_title = translate_text( title, source='en', target='fr')
        title = translated_title.lower()

        print(publish_date)
        print(title)
        
        tenders.append({
            "title": title,
            "publication_date": publish_date,
            "expiration_date": None,
            "link": l,
            "country": "Lesotho"
        })

    except Exception as e:
        print(f"Error extracting the title for {l}: {e}")

In [ ]:
driver.quit()

In [ ]:
df = pd.DataFrame(tenders)
df.to_excel("../2_scraped_files/Lesotho.xlsx", index=False) # modify